## Import des Librairies

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import perceval as pcvl
import sklearn.svm
import torch
from sklearn.decomposition import PCA
import merlin
from pathlib import Path
import os

## Class result

Utile pour garder les résultats, pas sûr que ce soit vraiment la meilleure manière de faire

In [ ]:
class result():
    def __init__(self, g, var_FQK, var_RBF, var_RBF_order_2, F,eta_max, ROC_AUC):
        self.var_FQK = var_FQK
        self.var_RBF = var_RBF
        self.var_RBF_order_2 = var_RBF_order_2
        self.g = g
        self.F = F
        self.eta_max = eta_max
        self.ROC_AUC = ROC_AUC


## Préprocessing

### Importation du dataset

In [ ]:

def charger_npz_torch(chemin_fichier, est_image=True):
    """
    Charge un fichier .npz et le convertit en tenseur PyTorch.
    """
    # 1. Chargement de l'archive NumPy
    archive = np.load(chemin_fichier)
   
    # Les fichiers .npz sont comme des dictionnaires. On récupère le premier tableau.
    nom_tableau = archive.files[0]
    donnees_np = archive[nom_tableau]
   
    # 2. Conversion en tenseur PyTorch
    tenseur = torch.from_numpy(donnees_np)
   
    # 3. Formatage spécifique selon que ce soit une image ou un label
    if est_image:
        # Conversion en float32, normalisation (0 à 1) et ajout du canal (B, 1, H, W)
        tenseur = tenseur.float() / 255.0
        tenseur = tenseur.unsqueeze(1)
    else:
        # Forçage en type Long pour les labels
        tenseur = tenseur.long()
       
    return tenseur

path = Path(__file__).resolve()
root = path.parents[3]
dossier_kmnist = "Users" / "TancredeTONINI" / "Documents" / "GitHub" / "reproduced_paper"  / "data" / "Bandwidth_tunning" / "kmnist"

# Noms exacts d'après votre capture d'écran
chemin_train_imgs = dossier_kmnist / 'kmnist-train-imgs.npz'
chemin_train_labels = dossier_kmnist / 'kmnist-train-labels.npz'
chemin_test_imgs = dossier_kmnist / 'kmnist-test-imgs.npz'
chemin_test_labels = dossier_kmnist / 'kmnist-test-labels.npz'

# --- CHARGEMENT DES DONNÉES ---
X_train = charger_npz_torch(chemin_train_imgs, est_image=True)
y_train = charger_npz_torch(chemin_train_labels, est_image=False)

X_test = charger_npz_torch(chemin_test_imgs, est_image=True)
y_test = charger_npz_torch(chemin_test_labels, est_image=False)

masque_train = (y_train == 2) | (y_train == 8)
masque_test = (y_test == 2) | (y_test == 8)

X_train = X_train[masque_train]
y_train = y_train[masque_train]

X_test = X_test[masque_test]
y_test = y_test[masque_test]

# --- VÉRIFICATION ---
print("--- Formats des Tenseurs K-MNIST ---")
print(f"X_train : {X_train.shape} | Type : {X_train.dtype}")
print(f"y_train : {y_train.shape} | Type : {y_train.dtype}")




In [ ]:
nb_images = 10

fig,axes = plt.subplots(2, nb_images//2, figsize=(15, 3))
axes = axes.flatten()
for i in range(nb_images):
    image_tensor = X_train[i].squeeze(0)  # Retirer la dimension du canal
    label_tensor = y_train[i]

    axes[i].imshow(image_tensor, cmap='gray')

    axes[i].set_title(f"Label: {label_tensor.item()}")
    axes[i].axis('off')

plt.tight_layout()
plt.show()

### Reduction du Dataset + PCA

In [ ]:
def subset_PCA(X_train, y_train, X_test, y_test, nb_train, nb_test, dim = -1, seed = 42):
    """Extrait un sous-ensemble des données et applique PCA pour réduire la dimensionnalité."""
    
    torch.manual_seed(seed)  # Pour la reproductibilité
    
    indices_train = torch.randperm(X_train.size(0))[:nb_train]
    indices_test = torch.randperm(X_test.size(0))[:nb_test]
   
    X_train_subset = X_train[indices_train].view(nb_train, -1).numpy()  # Aplatir les images
    y_train_subset = y_train[indices_train]
    X_test_subset = X_test[indices_test].view(nb_test, -1).numpy()  # Aplatir les images
    y_test_subset = y_test[indices_test]
    
    if dim != -1:
        # Appliquer PCA pour réduire à 'dim' dimensions
        pca = PCA(n_components=dim)
        X_train = pca.fit_transform(X_train_subset)
        X_test = pca.transform(X_test_subset)
    
    return torch.from_numpy(X_train).float(), y_train_subset, torch.from_numpy(X_test).float(), y_test_subset

## Calcul des métriques

### Calcul de $g(\textbf{K}_C,\textbf{K}_Q)$

In [ ]:
def matrix_sqrt(A): 
    """ Calcule la racine carrée matricielle d'une matrice symétrique semi-définie positive. """ 
    # Décomposition en valeurs propres (A = V * L * V^T) 
    L, V = torch.linalg.eigh(A)
    # Parfois, les erreurs de précision numérique créent des valeurs propres 
    # très légèrement négatives (ex: -1e-15). On les force à être positives ou nulles. 
    L = torch.clamp(L, min=0.0) 
    # Reconstruction : V * sqrt(L) * V^T 
    return V @ torch.diag(torch.sqrt(L)) @ V.T

def calculate_g(K1, K2): 
    """ Calcule g(K1, K2) avec lambda = 0. """ 
    # 1. Calcul de l'inverse de K2
    # Utilisation de pseudo-inverse (pinv) est plus sûr que inv()
    # car une matrice de noyau (K2) peut avoir un déterminant proche de 0. 
    K2_inv = torch.linalg.pinv(K2)
    # 2. Calcul de la racine carrée matricielle de K1 
    sqrt_K1 = matrix_sqrt(K1) 
    # 3. Produit matriciel central : sqrt(K1) @ K2^-1 @ sqrt(K1) 
    inner_matrix = sqrt_K1 @ K2_inv @ sqrt_K1 
    # 4. Norme spectrale (la plus grande valeur singulière, équivalente à ord=2) 
    spectral_norm = torch.linalg.matrix_norm(inner_matrix, ord=2) 
    # 5. Racine carrée finale de la formule 
    g = torch.sqrt(spectral_norm) 
    return g

### Calcul de $\eta_{max}$

In [ ]:
def calculate_eta_max(K):
    L,V = torch.linalg.eigh(K)
    eta_max = L[-1]
    return eta_max

### Calcul de $F(\textbf{K}_C,\textbf{K}_Q)$

In [ ]:
def calculate_kernel_distance_F(K_C, K_Q):
   """
   Calcule F(K_C, K_Q), la distance relative de Frobenius entre deux matrices de noyau.
   K_C : Tenseur PyTorch représentant la matrice de noyau classique.
   K_Q : Tenseur PyTorch représentant la matrice de noyau quantique.
   """
   # 1. Calcul du numérateur : Norme de Frobenius de la différence
   numerateur = torch.linalg.matrix_norm(K_C - K_Q, ord='fro')

   # 2. Calcul du dénominateur : Norme de Frobenius de K_Q
   denominateur = torch.linalg.matrix_norm(K_Q, ord='fro')

   # 3. Ratio final
   F = numerateur / denominateur

   return F

## Fonction d'entrainement de modèle

In [ ]:
def train(X_train, y_train_1D, X_test, y_test_1D, bandwidth = 1.0, n_modes = None, redundancy = False):

    X_train = X_train * bandwidth
    X_test = X_test * bandwidth

    if n_modes == None:
        n_modes = X_train.shape[1] + 1
    if n_modes < X_train.shape[1] + 1:
        raise ValueError(f"n_modes must be at least {X_train.shape[1] + 1} for the given input size.")


    builder = merlin.CircuitBuilder(n_modes=n_modes)
    builder.add_entangling_layer(trainable=True, model="mzi", name="left")
    builder.add_angle_encoding(modes=[i for i in range(X_train.shape[1])], name="phi")
    builder.add_entangling_layer(trainable=True, model="mzi", name="right")

    feature_map = merlin.FeatureMap(builder=builder, input_size=X_train.shape[1], input_parameters="phi")

    fidelity_kernel = merlin.FidelityKernel(
        feature_map=feature_map,
        input_state=[1 - (i % 2) for i in range(n_modes)],  # alternating photons for n_modes
        computation_space=merlin.ComputationSpace.FOCK,
    )

    svc = sklearn.svm.SVC(kernel="precomputed")

    K_train = fidelity_kernel(X_train)
    K_test = fidelity_kernel(X_test, X_train)


    svc.fit(K_train.detach().numpy(), y_train_1D.detach().numpy())

    #Calcul du kernel RBF
    distances = torch.cdist(X_train, X_train, p=2)
    distances_carré = distances ** 2
    K_rbf = torch.exp(-distances_carré)

    #Calcul du kernel RBF ordre 2
    distances = torch.cdist(X_train, X_train, p=2)
    z = distances ** 2
    K_rbf_order_2 = 1.0 - z + 0.5*(z**2)


    F = calculate_kernel_distance_F(K_train, K_rbf)
    eta_max = calculate_eta_max(K_train)
    ROC_AUC = sklearn.metrics.roc_auc_score(y_test_1D.detach().numpy(), svc.decision_function(K_test.detach().numpy()))

    return result(calculate_g(K_train,K_rbf).item(), K_train.var(correction=False).item(), K_rbf.var(correction=False).item(), K_rbf_order_2.var(correction=False).item(), F.item(), eta_max.item(), ROC_AUC)

In [ ]:
# Bandwidths (logarithmically spaced)
MIN,MAX,NB_Points = -3,3,60

# Stockage des résultats pour chaque métrique
x,y_g,y_FQK,y_RBF,y_RBF_order_2,y_F,y_eta_max,y_ROC_AUC = np.logspace(MIN, MAX, NB_Points),np.zeros(NB_Points),np.zeros(NB_Points),np.zeros(NB_Points),np.zeros(NB_Points),np.zeros(NB_Points),np.zeros(NB_Points),np.zeros(NB_Points)

# Size of the training and testing datasets
NB_TRAIN = 320
NB_TEST = 80

#SEEDS (for reproducibility)
SEEDS = [1, 19, 42, 67, 100, 2036] 


In [ ]:
for seed in SEEDS:
    print(f"\n--- Seed: {seed} ---")
    X_train, y_train, X_test, y_test = subset_PCA(X_train, y_train, X_test, y_test, nb_train=NB_TRAIN, nb_test=NB_TEST, dim = 4, seed = seed)
    for i in range(NB_Points):
        print("=",end="")
        res = train(X_train, y_train, X_test, y_test, bandwidth=x[i])
        y_g[i] += res.g
        y_FQK[i] += res.var_FQK
        y_RBF[i] += res.var_RBF
        y_RBF_order_2[i] += res.var_RBF_order_2
        y_F[i] += res.F
        y_eta_max[i] += res.eta_max
        y_ROC_AUC[i] += res.ROC_AUC

In [ ]:
y_g_avg = y_g / len(SEEDS)
y_FQK_avg = y_FQK / len(SEEDS)
y_RBF_avg = y_RBF / len(SEEDS)
y_RBF_order_2_avg = y_RBF_order_2 / len(SEEDS)
y_F_avg = y_F / len(SEEDS)
y_eta_max_avg = y_eta_max / len(SEEDS)
y_ROC_AUC_avg = y_ROC_AUC / len(SEEDS)

## Affichage des résultats

In [ ]:
# Création de la figure et d'une grille de 6 graphiques
# figsize=(15, 5) permet d'avoir une image bien large
fig, axes = plt.subplots( 2, 3, figsize=(15, 10))

# ==========================================
# Subplot 1 : Les Variances (FQK et RBF)
# ==========================================
axes[0,0].loglog(x, y_FQK_avg, label="FQK", marker='o', linestyle='-')
axes[0,0].loglog(x, y_RBF, label="RBF", marker='s', linestyle='-')

axes[0,0].set_title("Variances des noyaux")
axes[0,0].set_xlabel(r"Bandwidth $c$")
axes[0,0].set_ylabel(r"$Var_D[\mathbf{K}]$")
axes[0,0].legend() # Affiche la légende pour différencier les 3 courbes
axes[0,0].grid(True, which="both", ls="--", alpha=0.5) # Ajoute une grille lisible en log

# ==========================================
# Subplot 2 : La métrique g
# ==========================================
axes[0,1].loglog(x, y_g_avg, color='purple', marker='d', linestyle='-')

axes[0,1].set_title(r"Métrique $g$")
axes[0,1].set_xlabel(r"Bandwidth $c$")
axes[0,1].set_ylabel(r"$g(\mathbf{K}_C, \mathbf{K}_Q)$")
axes[0,1].grid(True, which="both", ls="--", alpha=0.5)

# ==========================================
# Subplot 3 : La distance F
# ==========================================
axes[1,0].loglog(x, y_F_avg, color='green', marker='v', linestyle='-')

axes[1,0].set_title("Distance de Frobenius")
axes[1,0].set_xlabel(r"Bandwidth $c$")
axes[1,0].set_ylabel(r"$F(\mathbf{K}_C, \mathbf{K}_Q)$")
axes[1,0].grid(True, which="both", ls="--", alpha=0.5)

# ==========================================
# Subplot 4 : eta_max
# ==========================================
axes[1,1].loglog(x, y_eta_max_avg, color='orange', marker='^', linestyle='-')

axes[1,1].set_title(r"$\eta_{max}$")
axes[1,1].set_xlabel(r"Bandwidth $c$")
axes[1,1].set_ylabel(r"$\eta_{max(K)}$")
axes[1,1].grid(True, which="both", ls="--", alpha=0.5)


# ==========================================
# Subplot 5 : ROC AUC
# ==========================================
axes[0,2].semilogx(x, y_ROC_AUC_avg, color='red', marker='s', linestyle='-')

axes[0,2].set_title(r"ROC AUC score")
axes[0,2].set_xlabel(r"Bandwidth $c$")
axes[0,2].set_ylabel("roc auc score")
axes[0,2].grid(True, which="both", ls="--", alpha=0.5)

# ==========================================
# Subplot 6 : VAR(RBF_order_2)
# ==========================================
axes[1,2].loglog(x, y_RBF_order_2_avg, color='blue', marker='x', linestyle='-')

axes[1,2].set_title("Variance du noyau RBF (ordre 2)")
axes[1,2].set_xlabel(r"Bandwidth $c$")
axes[1,2].set_ylabel("Variance")
axes[1,2].grid(True, which="both", ls="--", alpha=0.5)

# ==========================================
# Affichage propre
# ==========================================
fig.suptitle(r"kMNIST28 $(N = , PCA = 4, Seeds = 5)$")
# tight_layout empêche les titres et les labels de se chevaucher
plt.tight_layout() 
plt.show()